<span style="float: left;padding: 1.3em">![logo](Ligo.png)</span>
<span style="float: right;padding: 1.3em">![logo](Zewail-City.png)</span>

---
# Challenge #4 (_Advanced_)

In [ ]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

In [ ]:
# Main Libraries
from pycbc.frame import read_frame
from pycbc.filter import highpass, matched_filter
from pycbc.psd import welch, interpolate
from pycbc.types import FrequencySeries
from pycbc.waveform import get_td_waveform
import matplotlib.pyplot as plt
import numpy as np
import gc
import bilby
from bilby.gw.conversion import convert_to_lal_binary_black_hole_parameters

### Use the data file challenge3.gwf with channels H1:CHALLENGE3 and L1:CHALLENGE3.
### These are real LIGO data from O2, though we've adjusted the time labels and added some simulated signals.
### Any simulated signals have been added to both the H1 and L1 data
### All simulated signals have spin = 0 and m1 = m2, with m1 somewhere in the range 10-50 solar masses

In [ ]:
# Load both H1 and L1 data
file_name = "challenge3.gwf"
h1_channel = "H1:CHALLENGE3"
l1_channel = "L1:CHALLENGE3"

h1_data = read_frame(file_name, h1_channel)
l1_data = read_frame(file_name, l1_channel)

print("H1 - Sampling rate:", h1_data.sample_rate, "Hz, Duration:", h1_data.duration, "s")
print("L1 - Sampling rate:", l1_data.sample_rate, "Hz, Duration:", l1_data.duration, "s")

In [ ]:
# Plot both detectors' raw data
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(h1_data.sample_times, h1_data)
axes[0].set_ylabel("Strain")
axes[0].set_title("H1 - Raw Data")
axes[0].grid()

axes[1].plot(l1_data.sample_times, l1_data)
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Strain")
axes[1].set_title("L1 - Raw Data")
axes[1].grid()

plt.tight_layout()
plt.show()

In [ ]:
# Filter and crop both datasets
def filter_and_crop(data, f_low=30.0, crop_dur=90):
    """Filter with highpass and crop edges"""
    data_hp = highpass(data, f_low)
    crop_samples = int(crop_dur * data.sample_rate)
    return data_hp[crop_samples:-crop_samples]

h1_filtered = filter_and_crop(h1_data)
l1_filtered = filter_and_crop(l1_data)

print(f"Filtered H1 duration: {h1_filtered.duration:.1f} s")
print(f"Filtered L1 duration: {l1_filtered.duration:.1f} s")

In [ ]:
# Compute and plot FFT for both detectors

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# H1 FFT
h1_fft = h1_data.to_frequencyseries()
axes[0].loglog(h1_fft.sample_frequencies, abs(h1_fft), alpha=0.7, color='blue')
axes[0].set_ylabel("Amplitude")
axes[0].set_title("H1 - FFT Amplitude Spectrum")
axes[0].set_xlim(10, h1_data.sample_rate/2)
axes[0].grid(which='both', alpha=0.3)

# L1 FFT
l1_fft = l1_data.to_frequencyseries()
axes[1].loglog(l1_fft.sample_frequencies, abs(l1_fft), alpha=0.7, color='red')
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Amplitude")
axes[1].set_title("L1 - FFT Amplitude Spectrum")
axes[1].set_xlim(10, l1_data.sample_rate/2)
axes[1].grid(which='both', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Overlay comparison
plt.figure(figsize=(14, 6))
plt.loglog(h1_fft.sample_frequencies, abs(h1_fft), alpha=0.7, label='H1', color='blue')
plt.loglog(l1_fft.sample_frequencies, abs(l1_fft), alpha=0.7, label='L1', color='red')
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.title("FFT Amplitude Spectrum - Both Detectors (Overlay)")
plt.xlim(10, h1_data.sample_rate/2)
plt.legend()
plt.legend(loc='upper right')  # Specify location explicitly
plt.grid(which='both', alpha=0.3)
plt.show()

In [ ]:
# Print some statistics
print(f"H1 - Frequency resolution: {h1_fft.delta_f:.6f} Hz")
print(f"L1 - Frequency resolution: {l1_fft.delta_f:.6f} Hz")
print(f"H1 - Nyquist frequency: {h1_data.sample_rate/2:.1f} Hz")
print(f"L1 - Nyquist frequency: {l1_data.sample_rate/2:.1f} Hz")

# Find dominant frequency components (top 5 peaks)
h1_amp = abs(h1_fft)
h1_freq_mask = (h1_fft.sample_frequencies > 20) & (h1_fft.sample_frequencies < 512)
h1_amp_masked = h1_amp[h1_freq_mask]
h1_freqs_masked = h1_fft.sample_frequencies[h1_freq_mask]
h1_top_indices = np.argsort(h1_amp_masked)[-5:][::-1]

print("\nH1 - Top 5 frequency components (20-512 Hz):")
for idx in h1_top_indices:
    freq = h1_freqs_masked[idx]
    amp = h1_amp_masked[idx]
    print(f"  {freq:.2f} Hz: amplitude = {amp:.2e}")

l1_amp = abs(l1_fft)
l1_freq_mask = (l1_fft.sample_frequencies > 20) & (l1_fft.sample_frequencies < 512)
l1_amp_masked = l1_amp[l1_freq_mask]
l1_freqs_masked = l1_fft.sample_frequencies[l1_freq_mask]
l1_top_indices = np.argsort(l1_amp_masked)[-5:][::-1]

print("\nL1 - Top 5 frequency components (20-512 Hz):")
for idx in l1_top_indices:
    freq = l1_freqs_masked[idx]
    amp = l1_amp_masked[idx]
    print(f"  {freq:.2f} Hz: amplitude = {amp:.2e}")

In [ ]:
# Calculate PSDs for both detectors
h1_psd = welch(h1_filtered, seg_len=int(8*h1_filtered.sample_rate), 
               seg_stride=int(4*h1_filtered.sample_rate))
l1_psd = welch(l1_filtered, seg_len=int(8*l1_filtered.sample_rate), 
               seg_stride=int(4*l1_filtered.sample_rate))

In [ ]:
# Plot PSDs
plt.figure(figsize=(12, 6))
plt.loglog(h1_psd.sample_frequencies, h1_psd, label='H1', alpha=0.7)
plt.loglog(l1_psd.sample_frequencies, l1_psd, label='L1', alpha=0.7)
plt.xlabel("Frequency (Hz)")
plt.ylabel("PSD (1/Hz)")
plt.title("Power Spectral Density - Both Detectors")
plt.xlim(20, h1_filtered.sample_rate/2)
plt.legend()
plt.grid(which='both', alpha=0.3)
plt.show()

In [ ]:
# Function to search for signals with different masses
def search_for_signal(data, psd, mass, f_low=30.0):
    """Generate template and calculate SNR for given mass"""
    hp, _ = get_td_waveform(
        approximant='SEOBNRv4_opt',
        mass1=mass,
        mass2=mass,
        spin1z=0,
        spin2z=0,
        delta_t=1.0/data.sample_rate,
        f_lower=f_low
    )
    
    hp.resize(len(data))
    psd_interp = interpolate(psd, hp.delta_f)
    snr = matched_filter(hp, data, psd=psd_interp, low_frequency_cutoff=f_low)
    
    return snr, hp

# Search over mass range: 10, 15, 20, 25, 30, 35, 40, 45, 50 solar masses
masses_to_search = [10, 15, 20, 25, 30, 35, 40, 45, 50]

### Identify as many signals as you can. Watch out! These are real data, and so glitches may be present. For each signal you find, list:
- The merger time
- The SNR
- estimate of the component masses

In [ ]:
# Perform matched filtering search for H1
h1_results = {}

for mass in masses_to_search:
    snr, template = search_for_signal(h1_filtered, h1_psd, mass)
    peak_idx = np.argmax(abs(snr))
    peak_snr = abs(snr[peak_idx])
    peak_time = snr.sample_times[peak_idx]
    
    h1_results[mass] = {
        'snr_series': snr,
        'peak_snr': peak_snr,
        'peak_time': peak_time,
        'template': template
    }
    
    print(f"H1 - Mass {mass} Msun: SNR = {peak_snr:.2f}, Time = {peak_time:.2f} s")

In [ ]:
# Perform matched filtering search for L1
l1_results = {}

for mass in masses_to_search:
    snr, template = search_for_signal(l1_filtered, l1_psd, mass)
    peak_idx = np.argmax(abs(snr))
    peak_snr = abs(snr[peak_idx])
    peak_time = snr.sample_times[peak_idx]
    
    l1_results[mass] = {
        'snr_series': snr,
        'peak_snr': peak_snr,
        'peak_time': peak_time,
        'template': template
    }
    
    print(f"L1 - Mass {mass} Msun: SNR = {peak_snr:.2f}, Time = {peak_time:.2f} s")

In [ ]:
# Plot only SNR time series for masses with SNR > 8
def plot_significant_snrs(results, detector_name, snr_threshold=8.0):
    """Plot SNR time series only for significant detections"""
    
    # Filter to only significant masses
    significant_masses = [m for m in results.keys() 
                         if results[m]['peak_snr'] > snr_threshold]
    
    if len(significant_masses) == 0:
        print(f"No significant detections in {detector_name}")
        return
    
    # Create subplot grid
    n_plots = len(significant_masses)
    n_cols = min(3, n_plots)
    n_rows = (n_plots + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    if n_plots == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    downsample_factor = 100
    
    for idx, mass in enumerate(significant_masses):
        snr = results[mass]['snr_series']
        
        # Downsample
        times_ds = snr.sample_times[::downsample_factor]
        snr_ds = abs(snr)[::downsample_factor]
        
        axes[idx].plot(times_ds, snr_ds, linewidth=0.8)
        axes[idx].set_title(f"{detector_name} - {mass} Msun (SNR={results[mass]['peak_snr']:.1f})")
        axes[idx].set_xlabel("Time (s)")
        axes[idx].set_ylabel("SNR")
        axes[idx].grid(alpha=0.3)
        axes[idx].axhline(y=snr_threshold, color='r', linestyle='--', alpha=0.5)
        
        del times_ds, snr_ds
    
    # Hide unused subplots
    for idx in range(len(significant_masses), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.suptitle(f"{detector_name} - Significant Detections (SNR > {snr_threshold})", 
                 y=1.001, fontsize=14)
    plt.show()
    gc.collect()

In [ ]:
# Plot for H1 detector
plot_significant_snrs(h1_results, 'H1')

In [ ]:
# Plot for L1 detector
plot_significant_snrs(l1_results, 'L1')

### Identify as many glitches as you can. Make a spectrogram of each one.

In [ ]:
# Identify coincident detections (signals present in both H1 and L1)
# Threshold: SNR > 8, time difference < 0.1s

def find_coincident_signals(h1_res, l1_res, snr_threshold=8.0, time_window=0.1):
    """Find signals detected in both detectors"""
    coincident = []
    
    for mass in h1_res.keys():
        h1_snr = h1_res[mass]['peak_snr']
        l1_snr = l1_res[mass]['peak_snr']
        h1_time = h1_res[mass]['peak_time']
        l1_time = l1_res[mass]['peak_time']
        
        time_diff = abs(h1_time - l1_time)
        
        if h1_snr > snr_threshold and l1_snr > snr_threshold and time_diff < time_window:
            network_snr = np.sqrt(h1_snr**2 + l1_snr**2)
            coincident.append({
                'mass': mass,
                'h1_snr': h1_snr,
                'l1_snr': l1_snr,
                'network_snr': network_snr,
                'h1_time': h1_time,
                'l1_time': l1_time,
                'time_diff': time_diff
            })
    
    return coincident

coincident_signals = find_coincident_signals(h1_results, l1_results)

In [ ]:
for sig in coincident_signals:
    print(f"\nMass: {sig['mass']} + {sig['mass']} Msun")
    print(f"  H1: SNR = {sig['h1_snr']:.2f}, Time = {sig['h1_time']:.3f} s")
    print(f"  L1: SNR = {sig['l1_snr']:.2f}, Time = {sig['l1_time']:.3f} s")
    print(f"  Network SNR = {sig['network_snr']:.2f}")
    print(f"  Time difference = {sig['time_diff']*1000:.1f} ms")

In [ ]:
# Look for glitches: high SNR peaks that are NOT coincident
def find_glitches(h1_res, l1_res, snr_threshold=8.0, time_window=0.1):
    """Find high SNR events that don't coincide between detectors"""
    h1_glitches = []
    l1_glitches = []
    
    for mass in h1_res.keys():
        h1_snr = h1_res[mass]['peak_snr']
        l1_snr = l1_res[mass]['peak_snr']
        h1_time = h1_res[mass]['peak_time']
        l1_time = l1_res[mass]['peak_time']
        
        time_diff = abs(h1_time - l1_time)
        
        # H1 glitch: high SNR in H1 but not in L1 or not coincident
        if h1_snr > snr_threshold and (l1_snr < snr_threshold or time_diff > time_window):
            h1_glitches.append({
                'mass': mass,
                'snr': h1_snr,
                'time': h1_time
            })
        
        # L1 glitch: high SNR in L1 but not in H1 or not coincident
        if l1_snr > snr_threshold and (h1_snr < snr_threshold or time_diff > time_window):
            l1_glitches.append({
                'mass': mass,
                'snr': l1_snr,
                'time': l1_time
            })
    
    return h1_glitches, l1_glitches

h1_glitches, l1_glitches = find_glitches(h1_results, l1_results)

In [ ]:
print(f"\nH1 Glitches: {len(h1_glitches)}")
for glitch in h1_glitches:
    print(f"  Mass {glitch['mass']} Msun: SNR = {glitch['snr']:.2f}, Time = {glitch['time']:.2f} s")

print(f"\nL1 Glitches: {len(l1_glitches)}")
for glitch in l1_glitches:
    print(f"  Mass {glitch['mass']} Msun: SNR = {glitch['snr']:.2f}, Time = {glitch['time']:.2f} s")

In [ ]:
# Create spectrograms around each glitch
def plot_glitch_spectrogram(data, glitch_time, detector_name, window=2.0):
    """Plot Q-transform around a glitch"""
    # Extract time window around glitch
    start_time = glitch_time - window/2
    end_time = glitch_time + window/2
    
    # Find indices
    start_idx = int((start_time - data.start_time) * data.sample_rate)
    end_idx = int((end_time - data.start_time) * data.sample_rate)
    
    if start_idx < 0:
        start_idx = 0
    if end_idx > len(data):
        end_idx = len(data)
    
    data_segment = data[start_idx:end_idx]
    data_whitened = data_segment.whiten(1, 1)
    
    times, freqs, qplane = data_whitened.qtransform(
        delta_t=0.001,
        logfsteps=200,
        qrange=(4, 64),
        frange=(20, 512)
    )
    
    plt.figure(figsize=(10, 6))
    plt.pcolormesh(times, freqs, qplane, cmap='viridis', shading='auto')
    plt.colorbar(label='Normalized energy')
    plt.xlabel("Time (s)")
    plt.ylabel("Frequency (Hz)")
    plt.title(f"{detector_name} Glitch at t={glitch_time:.2f} s")
    plt.yscale('log')
    plt.ylim(20, 512)
    plt.grid(alpha=0.3)
    plt.axvline(glitch_time, color='r', linestyle='--', alpha=0.7, label='Glitch time')
    plt.legend()
    plt.show()

In [ ]:
# Plot spectrograms for H1 glitches
print("H1 Glitch Spectrograms:")
for glitch in h1_glitches[:5]:  # Limit to first 5
    plot_glitch_spectrogram(h1_filtered, glitch['time'], 'H1')

In [ ]:
# Plot spectrograms for L1 glitches
print("\nL1 Glitch Spectrograms:")
for glitch in l1_glitches[:5]:  # Limit to first 5
    plot_glitch_spectrogram(l1_filtered, glitch['time'], 'L1')

In [ ]:
# Summary of detections
print(f"\nCoincident Signals (Real BBH): {len(coincident_signals)}")
for sig in coincident_signals:
    print(f"  • {sig['mass']}+{sig['mass']} Msun at t≈{sig['h1_time']:.2f}s, Network SNR={sig['network_snr']:.1f}")

print(f"\nPotential Glitches:")
print(f"  • H1: {len(h1_glitches)} events")
print(f"  • L1: {len(l1_glitches)} events")

### For each simulated BBH you found, use bilby to compute a posterior distribution for the mass. You can fix the spin and mass ratio to make this run faster.

In [ ]:
# Bilby parameter estimation for each detected signal
def run_bilby_pe(data_h1, data_l1, psd_h1, psd_l1, mass_estimate, merger_time, 
                 duration=4, sample_rate=2048):
    """Run Bilby parameter estimation with fixed mass ratio and spins"""
    
    # Segment data around merger
    start_time = merger_time - duration/2
    
    # Calculate exact number of samples needed
    n_samples = int(duration * sample_rate)
    
    # Extract segments from filtered data
    start_idx_h1 = int((start_time - data_h1.start_time) * data_h1.sample_rate)
    end_idx_h1 = start_idx_h1 + int(duration * data_h1.sample_rate)
    segment_h1 = data_h1[start_idx_h1:end_idx_h1]
    
    start_idx_l1 = int((start_time - data_l1.start_time) * data_l1.sample_rate)
    end_idx_l1 = start_idx_l1 + int(duration * data_l1.sample_rate)
    segment_l1 = data_l1[start_idx_l1:end_idx_l1]
    
    # Resample to target sample rate if needed
    if data_h1.sample_rate != sample_rate:
        from pycbc.filter import resample_to_delta_t
        segment_h1 = resample_to_delta_t(segment_h1, 1.0/sample_rate)
        segment_l1 = resample_to_delta_t(segment_l1, 1.0/sample_rate)
    
    # Ensure exact length match
    segment_h1_array = np.array(segment_h1.numpy())[:n_samples]
    segment_l1_array = np.array(segment_l1.numpy())[:n_samples]
    
    # Pad if necessary
    if len(segment_h1_array) < n_samples:
        segment_h1_array = np.pad(segment_h1_array, (0, n_samples - len(segment_h1_array)))
    if len(segment_l1_array) < n_samples:
        segment_l1_array = np.pad(segment_l1_array, (0, n_samples - len(segment_l1_array)))
    
    print(f"Segment H1 length: {len(segment_h1_array)}, Expected: {n_samples}")
    print(f"Segment L1 length: {len(segment_l1_array)}, Expected: {n_samples}")
    
    # Set up interferometers
    ifos = bilby.gw.detector.InterferometerList(['H1', 'L1'])
    
    for ifo, segment_array in zip(ifos, [segment_h1_array, segment_l1_array]):
        # Set strain data
        ifo.strain_data.set_from_time_domain_strain(
            time_domain_strain=segment_array,
            sampling_frequency=sample_rate,
            duration=duration,
            start_time=start_time
        )
        
        ifo.minimum_frequency = 30.0
        ifo.maximum_frequency = sample_rate / 2
    
    # Generate PSD from ASD (use analytic PSD for simplicity)
    for ifo in ifos:
        ifo.power_spectral_density = bilby.gw.detector.PowerSpectralDensity.from_aligo()
    
    # Set up priors - simplified for faster convergence
    priors = bilby.gw.prior.BBHPriorDict()
    
    # Use chirp mass which is better constrained
    chirp_mass_estimate = mass_estimate * (0.5**0.2)  # For equal mass
    priors['chirp_mass'] = bilby.core.prior.Uniform(
        chirp_mass_estimate * 0.9, chirp_mass_estimate * 1.1, name='chirp_mass'
    )
    priors['mass_ratio'] = bilby.core.prior.DeltaFunction(1.0)  # Fixed equal mass
    priors['a_1'] = 0.0  # Fixed: no spin
    priors['a_2'] = 0.0  # Fixed: no spin
    priors['tilt_1'] = 0.0
    priors['tilt_2'] = 0.0
    priors['phi_12'] = 0.0
    priors['phi_jl'] = 0.0
    priors['luminosity_distance'] = bilby.core.prior.PowerLaw(
        alpha=2, minimum=500, maximum=3000, name='luminosity_distance'
    )
    priors['geocent_time'] = bilby.core.prior.Uniform(
        merger_time - 0.05, merger_time + 0.05, name='geocent_time'
    )
    priors['ra'] = bilby.core.prior.Uniform(0, 2*np.pi, name='ra')
    priors['dec'] = bilby.core.prior.Cosine(name='dec')
    priors['theta_jn'] = bilby.core.prior.Sine(name='theta_jn')
    priors['psi'] = bilby.core.prior.Uniform(0, np.pi, name='psi')
    priors['phase'] = bilby.core.prior.Uniform(0, 2*np.pi, name='phase')
    
    # Waveform generator
    waveform_generator = bilby.gw.WaveformGenerator(
        duration=duration,
        sampling_frequency=sample_rate,
        frequency_domain_source_model=bilby.gw.source.lal_binary_black_hole,
        parameter_conversion=convert_to_lal_binary_black_hole_parameters,
        waveform_arguments={
            'waveform_approximant': 'IMRPhenomPv2',
            'reference_frequency': 50.0,
            'minimum_frequency': 30.0
        }
    )
    
    # Likelihood
    likelihood = bilby.gw.GravitationalWaveTransient(
        interferometers=ifos,
        waveform_generator=waveform_generator,
        priors=priors
    )
    
    # Run sampler with reduced settings for faster runtime
    result = bilby.run_sampler(
        likelihood=likelihood,
        priors=priors,
        sampler='dynesty',
        nlive=200,  # Reduced for faster runtime
        npool=1,
        injection_parameters=None,
        outdir='bilby_output',
        label=f'bbh_mass_{int(mass_estimate)}',
        clean=True,
        verbose=True
    )
    
    return result

In [ ]:
# Run Bilby PE for each coincident signal
if len(coincident_signals) > 0:
    # Run for first detected signal as example
    first_signal = coincident_signals[0]
    
    print(f"Running Bilby PE for {first_signal['mass']}+{first_signal['mass']} Msun signal...")
    
    result = run_bilby_pe(
        data_h1=h1_filtered,
        data_l1=l1_filtered,
        psd_h1=h1_psd,
        psd_l1=l1_psd,
        mass_estimate=first_signal['mass'],
        merger_time=first_signal['h1_time']
    )
    
    # Plot corner plot
    result.plot_corner()

---